# European Flags — EfficientNetB2 Final Training

This is the **cleaned training notebook**.

Removed from the original notebook:
- duplicate/near-duplicate investigation cells that were already completed
- repeated exact-duplicate checks
- EfficientNetB3 experiments
- failed/redundant B2 fine-tuning blocks
- repeated dataset recreation/model loading cells

Kept:
- PC ZIP upload
- dataset discovery and verification
- exact-duplicate cleanup
- fixed stratified 697/149/150 split
- EfficientNetB2 Phase 1
- the proven B2 polishing stage
- final accuracy / precision / recall / F1 / loss
- confusion matrix and training curves
- model + class names download

**Benchmark to beat:** previous B2 polished model = 88.00% test accuracy, 89.08% macro precision, 86.18% macro F1, 0.4292 test loss.


## 1. GPU and reproducibility

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
import zipfile
import os
import random
import shutil

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tf.config.experimental.enable_op_determinism()

if not tf.config.list_physical_devices("GPU"):
    print("WARNING: GPU is NOT available.")
else:
    print("GPU is available!")


## 2. Upload `Europian_Flags.zip` from your PC

In [ ]:
from google.colab import files

uploaded = files.upload()

zip_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".zip")
]

if not zip_files:
    raise FileNotFoundError("No ZIP file was uploaded.")

ZIP_PATH = Path("/content") / zip_files[0]
EXTRACT_PATH = Path("/content/Europian_Flags")

if EXTRACT_PATH.exists():
    shutil.rmtree(EXTRACT_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("ZIP:", ZIP_PATH)
print("Extracted to:", EXTRACT_PATH)


## 3. Find and verify the dataset

In [ ]:
expected_classes = {
    "Austria", "Belgium", "Bulgaria", "Croatia", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Holland", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania",
    "Luxembourg", "Malta", "Slovakia", "Slovenia", "South Cyprus",
    "Spain", "Sweden"
}

DATA_DIR = None

for folder in [EXTRACT_PATH] + list(EXTRACT_PATH.rglob("*")):
    if folder.is_dir():
        folder_names = {
            x.name for x in folder.iterdir()
            if x.is_dir()
        }
        matches = len(folder_names.intersection(expected_classes))

        if matches >= 20:
            DATA_DIR = folder
            break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not automatically find the European Flags dataset folder."
    )

classes = sorted([
    x.name for x in DATA_DIR.iterdir()
    if x.is_dir()
])

print("DATA_DIR:", DATA_DIR)
print("Number of classes:", len(classes))

if set(classes) != expected_classes:
    print("Warning: class folder names differ from the expected list.")

for i, cls in enumerate(classes):
    print(f"{i:2d} -> {cls}")


## 4. Build the image DataFrame

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

records = []

for class_name in classes:
    class_dir = DATA_DIR / class_name

    for image_path in class_dir.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            records.append({
                "filepath": str(image_path),
                "class_name": class_name
            })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("Total classes:", df["class_name"].nunique())
print("\nImages per class:")
print(df["class_name"].value_counts().sort_index())


## 5. Remove exact duplicate files

In [ ]:
import hashlib
from collections import defaultdict

def sha256_file(filepath):
    sha = hashlib.sha256()

    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha.update(chunk)

    return sha.hexdigest()

exact_hashes = defaultdict(list)

for _, row in df.iterrows():
    try:
        file_hash = sha256_file(row["filepath"])
        exact_hashes[file_hash].append({
            "filepath": row["filepath"],
            "class_name": row["class_name"]
        })
    except Exception as e:
        print("Error:", row["filepath"], e)

clean_records = []

for file_hash, items in exact_hashes.items():
    clean_records.append(items[0])

clean_df = pd.DataFrame(clean_records)

print("Original images:", len(df))
print("Clean images   :", len(clean_df))
print("Removed        :", len(df) - len(clean_df))

cross_class_exact = {
    h: items
    for h, items in exact_hashes.items()
    if len({item["class_name"] for item in items}) > 1
}

print("Cross-class exact duplicate groups:", len(cross_class_exact))

if len(cross_class_exact) > 0:
    raise ValueError(
        "Cross-class exact duplicates detected. Review them before training."
    )


## 6. Create labels and the fixed 697/149/150 split

In [ ]:
class_names = sorted(clean_df["class_name"].unique())

class_to_index = {
    name: i for i, name in enumerate(class_names)
}

clean_df["label"] = clean_df["class_name"].map(class_to_index)

NUM_CLASSES = len(class_names)

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(train_df) + len(val_df) + len(test_df))

if (len(train_df), len(val_df), len(test_df)) != (697, 149, 150):
    print("WARNING: split sizes differ from the original benchmark split.")


## 7. TensorFlow datasets

In [ ]:
IMG_SIZE = (260, 260)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        IMG_SIZE,
        method=tf.image.ResizeMethod.BICUBIC
    )

    image = tf.cast(image, tf.float32)

    return image, label


def create_dataset(dataframe, training=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("Datasets created.")


## 8. Data augmentation and class weights

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(factor=0.05),
    tf.keras.layers.RandomZoom(
        height_factor=0.10,
        width_factor=0.10
    ),
    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    ),
    tf.keras.layers.RandomContrast(factor=0.10),
], name="data_augmentation")

from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].values
)

class_weights = {
    int(i): float(weight)
    for i, weight in enumerate(weights)
}

print("Class weights:")
for i, weight in class_weights.items():
    print(f"{i:2d} | {class_names[i]:20s} | {weight:.3f}")


## 9. Build EfficientNetB2

In [ ]:
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras import layers

inputs = tf.keras.Input(
    shape=(260, 260, 3),
    name="image"
)

x = data_augmentation(inputs)

base_model = EfficientNetB2(
    include_top=False,
    weights="imagenet",
    input_shape=(260, 260, 3)
)

base_model.trainable = False

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.40)(x)

x = layers.Dense(
    256,
    activation="relu"
)(x)

x = layers.BatchNormalization()(x)
x = layers.Dropout(0.30)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="predictions"
)(x)

model = tf.keras.Model(
    inputs,
    outputs,
    name="European_Flags_EfficientNetB2"
)

model.summary()


## 10. Phase 1 — train the B2 classifier

In [ ]:
MODEL_DIR = Path("/content/Europian_Flags_Models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL = MODEL_DIR / "best_european_flags.keras"

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)


## 11. Phase 2 — initial B2 fine-tuning

In [ ]:
base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-5,
        weight_decay=1e-5
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("Phase 1 best validation loss:",
      min(history1.history["val_loss"]))
print("Phase 2 best validation loss:",
      min(history2.history["val_loss"]))
print("Phase 1 best validation accuracy:",
      max(history1.history["val_accuracy"]))
print("Phase 2 best validation accuracy:",
      max(history2.history["val_accuracy"]))


## 12. Load the best B2 champion BEFORE polishing

In [ ]:
model = tf.keras.models.load_model(BEST_MODEL)

print("Best B2 champion loaded.")
print("Input shape:", model.input_shape)

base_model = None

for layer in model.layers:
    if "efficientnet" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise RuntimeError("EfficientNetB2 backbone not found.")

print("Backbone:", base_model.name)


## 13. B2 polished fine-tuning — proven configuration

In [ ]:
base_model.trainable = True

# Freeze everything except the last 30 backbone layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNorm frozen
for layer in base_model.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

trainable_layers = sum(
    layer.trainable
    for layer in base_model.layers
)

print(
    f"Trainable layers: "
    f"{trainable_layers}/{len(base_model.layers)}"
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-6,
        weight_decay=1e-5
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ]
)

BEST_B2_POLISHED = (
    MODEL_DIR / "best_european_flags_B2_polished.keras"
)

callbacks_polish = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_B2_POLISHED),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.3,
        patience=3,
        min_lr=1e-8,
        verbose=1
    )
]

history_polish = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks_polish,
    verbose=1
)

print("Best polished validation loss:",
      min(history_polish.history["val_loss"]))
print("Best polished validation accuracy:",
      max(history_polish.history["val_accuracy"]))


## 14. Load the best polished model and evaluate

In [ ]:
model = tf.keras.models.load_model(
    BEST_B2_POLISHED
)

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print("\n" + "=" * 60)
print(" EUROPEAN FLAGS — B2 POLISHED FINAL TEST")
print("=" * 60)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4%}")
print("=" * 60)


## 15. Precision, Recall and F1

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)

macro_recall = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

weighted_f1 = f1_score(
    y_true, y_pred, average="weighted", zero_division=0
)

print("=" * 65)
print(" EUROPEAN FLAGS — FINAL PERFORMANCE")
print("=" * 65)
print(f"Accuracy         : {accuracy:.4%}")
print(f"Macro Precision  : {macro_precision:.4%}")
print(f"Macro Recall     : {macro_recall:.4%}")
print(f"Macro F1         : {macro_f1:.4%}")
print(f"Weighted F1      : {weighted_f1:.4%}")
print(f"Test Loss        : {test_loss:.4f}")
print("=" * 65)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


## 16. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(18, 16))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("European Flags — Confusion Matrix", fontsize=18)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 17. Training curves

In [ ]:
acc = (
    history1.history["accuracy"]
    + history2.history["accuracy"]
    + history_polish.history["accuracy"]
)

val_acc = (
    history1.history["val_accuracy"]
    + history2.history["val_accuracy"]
    + history_polish.history["val_accuracy"]
)

loss = (
    history1.history["loss"]
    + history2.history["loss"]
    + history_polish.history["loss"]
)

val_loss = (
    history1.history["val_loss"]
    + history2.history["val_loss"]
    + history_polish.history["val_loss"]
)

epochs_range = range(1, len(acc) + 1)
phase1_end = len(history1.history["accuracy"])
phase2_end = phase1_end + len(history2.history["accuracy"])


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")

plt.axvline(
    x=phase1_end,
    linestyle="--",
    label="Phase 2 starts"
)

plt.axvline(
    x=phase2_end,
    linestyle="--",
    label="B2 polishing starts"
)

plt.title("European Flags — Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")

plt.axvline(
    x=phase1_end,
    linestyle="--",
    label="Phase 2 starts"
)

plt.axvline(
    x=phase2_end,
    linestyle="--",
    label="B2 polishing starts"
)

plt.title("European Flags — Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## 18. Save class names and final model

In [ ]:
import json

FINAL_MODEL_PATH = (
    MODEL_DIR / "FINAL_European_Flags_EfficientNetB2.keras"
)

model.save(FINAL_MODEL_PATH)

CLASS_NAMES_PATH = (
    MODEL_DIR / "class_names.json"
)

with open(
    CLASS_NAMES_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        class_names,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Final model:")
print(FINAL_MODEL_PATH)

print("\nClass names:")
print(CLASS_NAMES_PATH)


## 19. Download the final model and class names to your PC

In [ ]:
from google.colab import files

files.download(
    str(FINAL_MODEL_PATH)
)


In [ ]:
files.download(
    str(CLASS_NAMES_PATH)
)
